# Notebook 01 — Data Ingestion
Demonstrates fetching live NHC RSS feeds, advisory text, GIS links, and storm surge data.

**Data sources:**
- NHC RSS: https://www.nhc.noaa.gov/aboutrss.shtml
- NHC GIS: https://www.nhc.noaa.gov/gis/
- NOAA P-Surge: https://www.nhc.noaa.gov/surge/psurge.php

In [ ]:
import sys
sys.path.insert(0, '..')

from data_fetcher import (
    get_active_storms,
    fetch_storm_feeds,
    fetch_noaa_storm_surge,
    fetch_outlook_feeds,
)
from config import NHC_RSS_PATTERNS, NHC_OUTLOOK_FEEDS

print('Imports OK')

## 1. Check NHC Tropical Weather Outlooks (always available)
These feeds are available year-round, even when no named storms are active.

In [ ]:
outlooks = fetch_outlook_feeds()
for basin, text in outlooks.items():
    print(f'\n=== {basin.replace("_", " ").title()} Outlook ===')
    print(text[:500], '...' if len(text) > 500 else '')

## 2. Poll for Active Tropical Cyclones

In [ ]:
storms = get_active_storms()
print(f'Found {len(storms)} active storm(s)\n')
for s in storms:
    print(f"  Storm ID:   {s['storm_id']}")
    print(f"  Name:       {s['storm_type']} {s['name']}")
    print(f"  Basin:      {s['basin']}")
    print(f"  Advisory #: {s['advisory_number']}")
    print(f"  Published:  {s['published']}")
    print(f"  RSS URL:    {s['rss_url']}")
    print()

## 3. Fetch Advisory Text & GIS Links for First Storm
*(Skipped if no active storms — run during hurricane season)*

In [ ]:
if not storms:
    print('No active storms. Run during hurricane season (Atlantic: Jun 1 – Nov 30).')
else:
    storm = storms[0]
    print(f"Fetching feeds for {storm['storm_type']} {storm['name']}...")
    feeds = fetch_storm_feeds(storm)

    print('\n--- Advisory Text (first 1500 chars) ---')
    print(feeds['advisory_text'][:1500])

    print(f'\n--- GIS Links Found ({len(feeds["gis_links"])} total) ---')
    for link in feeds['gis_links']:
        print(' ', link)

    print(f'\n--- All RSS Entries ({len(feeds["all_entries"])} total) ---')
    for entry in feeds['all_entries']:
        print(f"  {entry['published']}: {entry['title']}")

## 4. Storm Surge Data

In [ ]:
if not storms:
    print('No active storms — surge data unavailable.')
else:
    surge = fetch_noaa_storm_surge(storms[0])
    print('--- Storm Surge Text ---')
    print(surge['surge_text'] or 'No surge text found in advisory.')
    print()
    if surge['surge_gdf'] is not None:
        print(f'Surge GIS loaded: {len(surge["surge_gdf"])} features')
        print(surge['surge_gdf'].head())
    else:
        print('No surge inundation shapefile available (only present for active surge threats).')

## 5. RSS Feed URL Structure Reference
Pattern: `https://www.nhc.noaa.gov/nhc_{basin}{n}.xml`

In [ ]:
print('NHC RSS Feed URL Patterns:')
for basin, pattern in NHC_RSS_PATTERNS.items():
    for n in range(1, 4):
        print(f'  {pattern.format(n=n)}')

print('\nNHC Outlook Feeds:')
for basin, url in NHC_OUTLOOK_FEEDS.items():
    print(f'  {url}')